In [ ]:
from datascience import *
import numpy as np
import warnings
warnings.filterwarnings("ignore")

%matplotlib inline
import matplotlib.pyplot as plots
plots.style.use('fivethirtyeight')
plots.rcParams["patch.force_edgecolor"] = True

### Use counts and bar charts for categorical distributions

In [ ]:
nba_2020 = Table.read_table('nba_salaries.csv').where('season', 2020)
nba_2020.show(6)

In [ ]:
nba_2020.group('position').barh('position', 'count')

In [ ]:
nba_2020.where('salary', are.above(2e7)).group('position').barh('position', 'count')

Exercise: Show the **percentage** of each players in each position instead of the count

In [ ]:
counts = nba_2020.group('position')
percentages = counts.with_column('percentage', 100 * counts.column('count') /  sum(counts.column('count')))
percentages.barh('position', 'percentage')

### Use binning for numerical distributions

In [ ]:
# Data from https://www.boxofficemojo.com/chart/ww_top_lifetime_gross/
top_movies = Table.read_table('top_movies_2026.csv')
top_movies.show(4)

In [ ]:
top_movies.set_format([1, 2, 3], CurrencyFormatter).show(4)

In [ ]:
top_movies.scatter(2, 3)

In [ ]:
top_movies.sort(3, descending=True).show(5)

#### **Task**: Visualize the distribution of movie ages

In [ ]:
ages = 2026 - top_movies.column('Year')
ages

In [ ]:
top_movies = top_movies.with_column('Age', ages)
top_movies

In [ ]:
top_movies.select('Title', 'Age').show(6)

In [ ]:
min(ages), max(ages)

- If you want to make equally sized bins, `np.arange()` is a great tool to help you.

In [ ]:
np.arange(0, max(ages)+10, 10)

In [ ]:
top_movies.hist('Age', bins = np.arange(0, max(ages)+10, 10), unit = 'Year')

- Otherwise, you can pick your own bins. These are just bins that we picked out.


In [ ]:
my_bins = make_array(0, 5, 10, 15, 25, 60)

In [ ]:
binned_data = top_movies.bin('Age', bins = my_bins)
binned_data

**Note:** The last "bin" does not include any observations!!

### Introducing the histogram and the area principle

In [ ]:
top_movies.hist('Age', bins = my_bins, unit = 'Year')

#### **Discussion Question**: Compare the bins $[0, 5)$ and $[15, 25)$. 

- Which one has more movies?
- Which one is more crowded?

### Challenge tasks

#### **Task**: Find the height of the $[15,25)$ bin in the histogram above.

$$\text{height} = \frac{\text{percent}}{\text{width}}$$

Add a column containing what percent of movies are in each bin (the **area** of each bin)

In [ ]:
binned_data = binned_data.with_column('Percent', 100*binned_data.column('Age count')/top_movies.num_rows)

In [ ]:
binned_data.show()

In [ ]:
percent = binned_data.where('bin', 15).column('Percent').item(0)

In [ ]:
width = 25-15
height = percent / width

In [ ]:
height

#### **Task**: Find the heights of the (rest of the) bins.

$$\text{height} = \frac{\text{percent}}{\text{width}}$$

Remember that the last row in the table does not represent a bin!

In [ ]:
height_table = binned_data.take(np.arange(binned_data.num_rows - 1))
height_table 

Remember `np.diff`?

In [ ]:
bin_widths = np.diff(binned_data.column('bin'))

In [ ]:
bin_widths

In [ ]:
height_table = height_table.with_column('Width', bin_widths)
height_table

In [ ]:
height_table = height_table.with_column('Height',
                                        height_table.column('Percent')/height_table.column('Width'))

In [ ]:
height_table

To check our work one last time, let's see if the numbers in the last column match the heights of the histogram:

In [ ]:
top_movies.hist('Age', bins = my_bins, unit = 'Year')